# Module 12: Performance Tuning & Spark Architecture

Understanding how Spark works **under the hood** is critical for interviews. Interviewers want to know you can write efficient code — not just code that works.

In this module you'll learn:
1. **Spark architecture** — driver, executors, tasks, stages
2. **Lazy evaluation & the DAG** — why transformations aren't executed immediately
3. **Partitions** — `repartition` vs `coalesce`
4. **Caching** — when to use `cache()`/`persist()` and when not to
5. **Broadcast joins** — avoiding shuffles for small tables
6. **Shuffles** — narrow vs wide transformations
7. **Reading query plans** — `.explain()`
8. **Common interview questions** — data skew, `mapPartitions`, optimization tips

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum, avg, round, broadcast, spark_partition_id
import time

spark = SparkSession.builder \
    .appName("Module 12 - Performance Tuning") \
    .master("local[*]") \
    .getOrCreate()

employees = spark.read.csv("../data/employees.csv", header=True, inferSchema=True)
departments = spark.read.csv("../data/departments.csv", header=True, inferSchema=True)
sales = spark.read.csv("../data/sales.csv", header=True, inferSchema=True)

print(f"Spark version: {spark.version}")
print(f"Default parallelism: {spark.sparkContext.defaultParallelism}")

---
## Concept 1: Spark Architecture

When you run a Spark application, here's what happens:

```
┌─────────────────────────────────────────────────────┐
│                    DRIVER                            │
│  Your code runs here. Creates the SparkSession,     │
│  builds the DAG, and sends tasks to executors.       │
└────────────────────┬────────────────────────────────┘
                     │ sends tasks
        ┌────────────┼────────────┐
        ▼            ▼            ▼
┌──────────┐  ┌──────────┐  ┌──────────┐
│ EXECUTOR │  │ EXECUTOR │  │ EXECUTOR │
│  Task 1  │  │  Task 2  │  │  Task 3  │
│  Task 4  │  │  Task 5  │  │  Task 6  │
│ (Cache)  │  │ (Cache)  │  │ (Cache)  │
└──────────┘  └──────────┘  └──────────┘
   Worker 1      Worker 2      Worker 3
```

Key concepts:
- **Driver** — your main program. Coordinates everything but doesn't process data.
- **Executors** — JVM processes on worker nodes that actually process data.
- **Task** — the smallest unit of work. One task processes one **partition** of data.
- **Stage** — a group of tasks that can run without a shuffle. A shuffle creates a new stage.
- **Job** — triggered by an **action** (like `.show()`, `.count()`, `.collect()`). One job = one or more stages.

**Interview tip**: When asked "how does Spark execute a query?", walk through: action triggers job → job splits into stages at shuffle boundaries → each stage runs tasks in parallel (one per partition).

---
## Concept 2: Lazy Evaluation & the DAG

Spark uses **lazy evaluation**: transformations (`.filter()`, `.select()`, `.join()`) don't execute immediately. Spark just records them in a **DAG (Directed Acyclic Graph)**.

Execution only happens when you call an **action** (`.show()`, `.count()`, `.collect()`, `.write`).

**Why?** So Spark can optimize the entire plan at once. It can:
- Push filters down closer to the data source (predicate pushdown)
- Combine multiple operations into one stage
- Skip reading columns you don't use (column pruning)

In [ ]:
# These transformations execute NOTHING — just build the plan
result = (
    sales
    .filter(col("amount") > 1000)
    .join(employees, on="employee_id")
    .groupBy("city")
    .agg(sum("amount").alias("total"))
)

print("No computation yet — just a plan.")
print(f"Type: {type(result)}")

# THIS triggers execution:
result.show()

In [ ]:
# .explain() shows the execution plan WITHOUT running it
# Read bottom-up: data flows from bottom (scan) to top (result)
result.explain()

In [ ]:
# Extended explain shows more detail — useful for interview discussions
result.explain(mode="formatted")

**Reading the plan**: Look for these keywords:
- `Scan` — reading from source
- `Filter` — row filtering (look for pushed-down filters near the Scan)
- `Project` — column selection
- `Exchange` / `Shuffle` — data moving between executors (expensive!)
- `HashAggregate` — groupBy aggregation
- `BroadcastHashJoin` — small table broadcast (good)
- `SortMergeJoin` — both tables shuffled (expensive)

#### Try It: Read a Query Plan

1. Build a query that joins `sales` with `departments` (through `employees`), groups by `department_name`, and sums `amount`
2. Call `.explain()` on the result
3. Can you identify the shuffle/exchange operations?

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
dept_revenue = (
    sales
    .join(employees, on="employee_id")
    .join(departments, on="department_id")
    .groupBy("department_name")
    .agg(round(sum("amount"), 2).alias("total_revenue"))
)

dept_revenue.explain()
print("---")
dept_revenue.show()

---
## Concept 3: Partitions — repartition vs coalesce

Data in Spark is split into **partitions**. Each partition is processed by one task on one core.

- **Too few partitions** → underutilize CPUs, possible memory issues
- **Too many partitions** → overhead from scheduling tiny tasks
- **Skewed partitions** → some tasks take much longer than others

Two ways to change partition count:

| | `repartition(n)` | `coalesce(n)` |
|---|---|---|
| Direction | Increase or decrease | **Only decrease** |
| Shuffle | **Yes** (full shuffle) | **No** (merges existing partitions) |
| Use when | Need even distribution, or increasing partitions | Reducing partitions before writing to disk |

**Interview answer**: Use `coalesce` to reduce partitions (no shuffle). Use `repartition` to increase partitions or to redistribute data evenly.

In [ ]:
# Check current partitions
print(f"Sales partitions: {sales.rdd.getNumPartitions()}")
print(f"Employees partitions: {employees.rdd.getNumPartitions()}")

# See which rows are in which partition
sales.withColumn("partition_id", spark_partition_id()) \
    .groupBy("partition_id").count().orderBy("partition_id").show()

In [ ]:
# Repartition — causes a full shuffle
sales_4 = sales.repartition(4)
print(f"After repartition(4): {sales_4.rdd.getNumPartitions()} partitions")

sales_4.withColumn("partition_id", spark_partition_id()) \
    .groupBy("partition_id").count().orderBy("partition_id").show()

In [ ]:
# Coalesce — merges partitions WITHOUT a shuffle
sales_2 = sales_4.coalesce(2)
print(f"After coalesce(2): {sales_2.rdd.getNumPartitions()} partitions")

# Compare explain plans
print("\nRepartition plan:")
sales_4.explain()

print("\nCoalesce plan:")
sales_2.explain()

In [ ]:
# Repartition by column — useful for even distribution by key
sales_by_region = sales.repartition(3, "region")

sales_by_region.withColumn("partition_id", spark_partition_id()) \
    .groupBy("partition_id", "region").count() \
    .orderBy("partition_id", "region").show()

#### Try It: Partition Exploration

1. Check how many partitions `employees` has
2. Repartition to 6 partitions and verify
3. Coalesce back to 2 partitions and verify
4. Compare the explain plans of steps 2 and 3

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
print(f"Original: {employees.rdd.getNumPartitions()} partitions")

emp_6 = employees.repartition(6)
print(f"After repartition(6): {emp_6.rdd.getNumPartitions()} partitions")

emp_2 = emp_6.coalesce(2)
print(f"After coalesce(2): {emp_2.rdd.getNumPartitions()} partitions")

print("\nRepartition plan:")
emp_6.explain()

print("\nCoalesce plan:")
emp_2.explain()

---
## Concept 4: Caching & Persistence

By default, Spark recomputes a DataFrame every time you use it (because of lazy evaluation). If you use a DataFrame multiple times, **cache it** to avoid redundant computation.

- `df.cache()` — stores in memory (same as `persist(StorageLevel.MEMORY_AND_DISK)`)
- `df.persist(level)` — choose storage level (MEMORY_ONLY, MEMORY_AND_DISK, DISK_ONLY, etc.)
- `df.unpersist()` — free the cached data

**When to cache**:
- DataFrame is used multiple times (e.g., train/test split, multiple aggregations)
- DataFrame is expensive to compute (complex joins, heavy transformations)

**When NOT to cache**:
- DataFrame is used only once
- DataFrame is very large (fills up memory, causes spilling)
- Data changes frequently

In [ ]:
# Build an expensive DataFrame
expensive_df = (
    sales
    .join(employees, on="employee_id")
    .join(departments, on="department_id")
    .filter(col("amount") > 500)
)

# Without caching — Spark recomputes the joins for EACH action
start = time.time()
count1 = expensive_df.count()
avg1 = expensive_df.agg(avg("amount")).collect()[0][0]
groups1 = expensive_df.groupBy("department_name").count().collect()
print(f"Without cache — 3 actions: {time.time() - start:.2f}s")

# With caching — computed once, reused from memory
expensive_df.cache()

start = time.time()
count2 = expensive_df.count()  # triggers computation and caching
avg2 = expensive_df.agg(avg("amount")).collect()[0][0]  # reads from cache
groups2 = expensive_df.groupBy("department_name").count().collect()  # reads from cache
print(f"With cache    — 3 actions: {time.time() - start:.2f}s")

# Clean up
expensive_df.unpersist()
print("\nCache cleared.")

In [ ]:
# Check what's cached in the Spark UI
from pyspark import StorageLevel

# Persist with a specific storage level
sales.persist(StorageLevel.MEMORY_AND_DISK)
sales.count()  # trigger caching

print(f"Is cached: {sales.is_cached}")
print(f"Storage level: {sales.storageLevel}")

sales.unpersist()
print(f"After unpersist: {sales.is_cached}")

#### Try It: Cache an Expensive DataFrame

1. Create a joined DataFrame: `sales` + `employees` + `departments`
2. Cache it
3. Run `.count()` to trigger caching
4. Verify it's cached with `.is_cached`
5. Run 2 more actions (e.g., `.groupBy().count()` and `.agg()`)
6. Unpersist when done

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
full = sales.join(employees, on="employee_id").join(departments, on="department_id")

full.cache()
print(f"Row count: {full.count()}")
print(f"Is cached: {full.is_cached}")

full.groupBy("department_name").count().show()
full.agg(round(avg("amount"), 2).alias("avg_amount")).show()

full.unpersist()
print(f"After unpersist: {full.is_cached}")

---
## Concept 5: Broadcast Joins

A normal join **shuffles** both DataFrames across the network so matching keys end up on the same executor. This is expensive for large datasets.

A **broadcast join** sends the entire small table to every executor. No shuffle needed for the large table — each executor joins its local partition with the full small table.

```
Normal join (shuffle):           Broadcast join:
┌─────┐    ┌─────┐              ┌─────┐    ┌─────┐
│Big A│───▶│Shuf-│              │Big A│    │Small│
└─────┘    │ fle │──▶ Join      │Part1│◀──│  B  │ (full copy)
┌─────┐    │     │              │Part2│◀──│  B  │ (full copy)
│Big B│───▶│     │              │Part3│◀──│  B  │ (full copy)
└─────┘    └─────┘              └─────┘    └─────┘
```

**Rule of thumb**: Broadcast the small table (< 10MB by default, configurable with `spark.sql.autoBroadcastJoinThreshold`).

Spark auto-broadcasts small tables, but you can force it with `broadcast()`.

In [ ]:
# Normal join — check the plan
normal_join = sales.join(employees, on="employee_id")
print("Normal join plan:")
normal_join.explain()

In [ ]:
# Explicit broadcast — force the small table to be broadcast
broadcast_join = sales.join(broadcast(departments), on="department_id")
print("Broadcast join plan:")
broadcast_join.explain()

In [ ]:
# Departments is tiny (6 rows) — perfect for broadcasting
# Look for "BroadcastHashJoin" in the plan (good!)
# vs "SortMergeJoin" or "ShuffledHashJoin" (expensive!)

optimized = (
    sales
    .join(employees, on="employee_id")
    .join(broadcast(departments), on="department_id")
    .groupBy("department_name")
    .agg(round(sum("amount"), 2).alias("total"))
)

optimized.explain()
optimized.show()

#### Try It: Broadcast Join

1. Join `sales` with `employees` (normal join — employees is 30 rows, small enough to broadcast)
2. Call `.explain()` and see the join strategy
3. Now wrap employees in `broadcast()` and compare the explain plan

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
print("Without broadcast hint:")
sales.join(employees, on="employee_id").explain()

print("\nWith broadcast hint:")
sales.join(broadcast(employees), on="employee_id").explain()

---
## Concept 6: Shuffles — Narrow vs Wide Transformations

A **shuffle** moves data across the network between executors. It's the most expensive operation in Spark.

| Narrow (no shuffle) | Wide (causes shuffle) |
|---|---|
| `select`, `filter`, `withColumn` | `groupBy`, `join`, `repartition` |
| `map`, `flatMap` | `reduceByKey`, `sortBy` |
| `union` | `distinct` |
| Each output partition depends on ONE input partition | Each output partition depends on MANY input partitions |

**Why shuffles are expensive**:
1. Data is serialized, sent over the network, and deserialized
2. Data is written to disk during the shuffle
3. Creates a new stage boundary (can't pipeline with prior operations)

**Interview tip**: When asked "how would you optimize this query?", first look for unnecessary shuffles — extra `groupBy`, `distinct`, or `repartition` calls.

In [ ]:
# Narrow transformations — no shuffle (one stage)
narrow = sales.filter(col("amount") > 1000).select("product", "amount")
print("Narrow (no shuffle):")
narrow.explain()

# Wide transformation — causes a shuffle (new stage)
wide = sales.groupBy("region").agg(sum("amount").alias("total"))
print("\nWide (shuffle):")
wide.explain()

In [ ]:
# Count shuffles in a complex query by counting "Exchange" in the plan
complex_query = (
    sales
    .join(employees, on="employee_id")
    .join(departments, on="department_id")
    .groupBy("department_name", "region")
    .agg(sum("amount").alias("total"))
    .orderBy(col("total").desc())
)

print("Complex query plan — look for 'Exchange' (shuffles):")
complex_query.explain()

---
## Concept 7: Common Interview Questions

### Data Skew

**Problem**: One partition has much more data than others. One task takes forever while others finish quickly.

**Example**: Joining on `city` when 90% of data is in "New York".

**Solutions**:
1. **Salting** — add a random number to the join key, splitting hot keys across partitions
2. **Broadcast** — if one side is small enough, broadcast it
3. **Adaptive Query Execution (AQE)** — Spark 3.x can auto-split skewed partitions

In [ ]:
# Detect skew — check partition sizes
from pyspark.sql.functions import rand, concat, lit

# Check data distribution by region
sales.groupBy("region").count().orderBy(col("count").desc()).show()

# If one region had 90% of data, that's skew.
# Salting example: add random suffix to break up hot keys
num_salts = 3
salted = sales.withColumn(
    "salted_region",
    concat(col("region"), lit("_"), (rand() * num_salts).cast("int"))
)

salted.select("region", "salted_region").show(10)

### mapPartitions vs map

- `map(func)` — applies function to each **row** individually
- `mapPartitions(func)` — applies function to each **partition** (iterator of rows)

`mapPartitions` is faster when you have expensive per-call setup (e.g., opening a database connection, loading a model). You set up once per partition instead of once per row.

In [ ]:
# mapPartitions example — process each partition as a batch
def process_partition(iterator):
    # Expensive setup would go here (DB connection, model load, etc.)
    results = []
    for row in iterator:
        results.append((row.employee_id, row.amount * 1.1))  # 10% markup
    return iter(results)

# Apply to RDD (mapPartitions works on RDDs)
marked_up = sales.rdd.mapPartitions(process_partition)
marked_up_df = marked_up.toDF(["employee_id", "marked_up_amount"])
marked_up_df.show(5)

### Quick Reference: Optimization Checklist

| Problem | Solution |
|---------|----------|
| Slow joins with small table | Use `broadcast()` |
| DataFrame used multiple times | `cache()` / `persist()` |
| Too many small output files | `coalesce(n)` before write |
| Data skew (one partition huge) | Salting, broadcast, or AQE |
| Reading more data than needed | Filter early, select only needed columns |
| Using CSV/JSON | Switch to Parquet (columnar, compressed) |
| UDFs in Python | Use built-in Spark functions instead (run in JVM, not Python) |
| `collect()` on large data | Use `.show()`, `.take()`, or write to file instead |

---
## Capstone Exercise

You have this slow query. Identify the problems and optimize it:

```python
# SLOW VERSION
result = (
    sales
    .join(employees, on="employee_id")
    .join(departments, on="department_id")
    .repartition(200)  # Is this needed?
)

# Used 3 times but not cached
result.groupBy("department_name").agg(sum("amount")).show()
result.groupBy("region").agg(sum("amount")).show()
result.count()
```

Optimize it by:
1. Removing the unnecessary `repartition`
2. Using `broadcast` for the small `departments` table
3. Caching since the result is used 3 times
4. Unpersisting when done

In [ ]:
# Your optimized code here


In [ ]:
# --- Capstone Solution ---

# OPTIMIZED VERSION
result = (
    sales
    .join(employees, on="employee_id")           # employees is small, Spark may auto-broadcast
    .join(broadcast(departments), on="department_id")  # force broadcast for departments
    # Removed unnecessary repartition(200) — it caused an extra shuffle for no benefit
).cache()  # Cache since we use it 3 times

# First action triggers computation + caching
result.groupBy("department_name").agg(round(sum("amount"), 2).alias("total")).show()

# These read from cache — no recomputation
result.groupBy("region").agg(round(sum("amount"), 2).alias("total")).show()
print(f"Total rows: {result.count()}")

# Clean up
result.unpersist()

print("\nOptimized plan:")
result.explain()

In [ ]:
spark.stop()

---
## What You Learned

- **Architecture**: Driver coordinates, executors process data in parallel
- **Lazy evaluation**: Transformations build a plan; actions execute it
- **`.explain()`**: Read query plans to find shuffles and join strategies
- **Partitions**: `repartition` (shuffle) vs `coalesce` (no shuffle)
- **Caching**: `cache()` DataFrames used multiple times; `unpersist()` when done
- **Broadcast joins**: Send small tables to all executors to avoid shuffles
- **Shuffles**: Wide transformations (groupBy, join, repartition) are expensive
- **Data skew**: Detect with partition size checks; fix with salting or broadcast

### Interview Cheat Sheet

| Question | Key Points |
|----------|------------|
| repartition vs coalesce? | repartition shuffles (increase/decrease), coalesce merges (decrease only, no shuffle) |
| When to cache? | DataFrame used multiple times AND expensive to compute |
| How to optimize a slow join? | Broadcast small table, check for skew, filter early |
| Narrow vs wide transformations? | Narrow = no shuffle (filter, select). Wide = shuffle (groupBy, join) |
| What causes data skew? | Uneven key distribution. Fix with salting or broadcast |
| Why Parquet over CSV? | Columnar, compressed, supports predicate pushdown |